In [3]:
from pathlib import Path
import os, subprocess

def get_project_root(max_up=6):
    try:
        root = subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip()
        if root:
            return Path(root)
    except Exception:
        pass
    p = Path.cwd()
    for _ in range(max_up):
        if (p / "data").exists() and (p / "code").exists():
            return p
        if (p / ".git").exists():
            return p
        p = p.parent
    return Path.cwd()

PROJECT_ROOT = get_project_root()
print("Project root:", PROJECT_ROOT)


Project root: /Users/liuq13/bhutan_climate_modeling


In [5]:
# --- Build numeric basin_id ↔ basin_name crosswalk ---

from pathlib import Path
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import re

# 1) Inputs
MAPPING = PROJECT_ROOT / "data/spatial/grid_mapping/era5_grid_to_polygons.parquet"
BASINS_DIR = PROJECT_ROOT / "data/boundaries/basins"

# Try to find a vector file in basins dir (shp or gpkg)
candidates = list(BASINS_DIR.glob("*.gpkg")) + list(BASINS_DIR.glob("*.shp"))
assert candidates, f"No vector file found in {BASINS_DIR}"
BASINS_PATH = candidates[0]
print("Using basins:", BASINS_PATH.name)

# 2) Load mapping points (only in-basin points)
mp = pd.read_parquet(MAPPING, columns=["grid_id","latitude","longitude","basin_id","outside_flag"])
mp = mp.loc[(~mp["outside_flag"]) & mp["basin_id"].notna()].copy()

# Use a small sample per basin to speed up the spatial join
sample = (mp.groupby("basin_id", group_keys=False)
            .apply(lambda d: d.sample(min(len(d), 50), random_state=42))
            .reset_index(drop=True))

gpoints = gpd.GeoDataFrame(sample,
                           geometry=[Point(xy) for xy in zip(sample["longitude"], sample["latitude"])],
                           crs="EPSG:4326")

# 3) Load basins polygons; pick a likely name column
basins = gpd.read_file(BASINS_PATH)
if basins.crs is None or basins.crs.to_epsg() != 4326:
    basins = basins.to_crs(4326)

name_candidates = ["basin_name","BASIN_NAME","name","Name","Basin","basin"]
name_col = next((c for c in name_candidates if c in basins.columns), None)
assert name_col is not None, f"Could not find a basin name column in {basins.columns.tolist()}"

# 4) Spatial join → recover basin_name for each numeric basin_id
joined = gpd.sjoin(gpoints, basins[[name_col, "geometry"]], how="left", predicate="within")
# For each numeric basin_id, take the most frequent basin_name
cross = (joined.groupby("basin_id")[name_col]
               .agg(lambda s: s.value_counts().index[0] if len(s.dropna()) else None)
               .reset_index()
               .rename(columns={name_col:"basin_name"}))

def slugify(s: str) -> str:
    return re.sub(r"[^a-z0-9_]+", "", re.sub(r"\s+", "_", s.lower())).strip("_") if isinstance(s, str) else s

cross["basin_slug"] = cross["basin_name"].map(slugify)

# 5) Save lookup
OUT = PROJECT_ROOT / "data/boundaries/processed/basin_lookup.csv"
OUT.parent.mkdir(parents=True, exist_ok=True)
cross.to_csv(OUT, index=False)
print("Wrote:", OUT)
display(cross)


Using basins: Basin boundary.shp
Wrote: /Users/liuq13/bhutan_climate_modeling/data/boundaries/processed/basin_lookup.csv


/var/folders/br/nr4k1vxj1_j7jxk17x7xr8n9g2q0k8/T/ipykernel_29990/2305399886.py:24: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sample = (mp.groupby("basin_id", group_keys=False)


,basin_id,basin_name,basin_slug
0,1.0,Aiechhu,aiechhu
1,3.0,Mangdechhu,mangdechhu
2,4.0,Jaldhakha,jaldhakha
3,5.0,Amochhu,amochhu
4,6.0,Wangchhu,wangchhu
5,7.0,Drangmechhu,drangmechhu
6,8.0,Punatsangchhu,punatsangchhu
7,9.0,Nyera_Amari,nyera_amari
8,10.0,Jomori,jomori
